In [ ]:

# ============================================================
# 2. Install dependencies
# ============================================================
!pip -q install -U "transformers==4.46.3" "nltk==3.9.1" "scikit-learn==1.5.2" "tqdm==4.66.6"

# Internet must be enabled in Kaggle.


In [ ]:
# ============================================================
# 3. Imports, reproducibility, GPU configuration
# ============================================================
import os, gc, math, random, re, json, zipfile, urllib.request, time, subprocess
from pathlib import Path
from collections import Counter, defaultdict

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

# -------------------- FINAL FULL JOINT CONFIG --------------------
NOTEBOOK_VERSION = "FINAL_PAPER_FAITHFUL_CAO2024"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 96
GLOSS_MAX_LEN = 128

# The paper reports 226,036 annotated SemCor examples.
# None means: use every successfully extracted SemCor example.
MAX_TRAIN_EXAMPLES = None
EXPECTED_SEMCOR_EXAMPLES = 226036

# Paper context batch size = 4.
CONTEXT_BATCH_SIZE = 4
MAX_ITERATIONS = 20
EPOCHS = 2   # the paper's maximum is 20 iterations; one epoch ~4h on a T4, so 20 cannot fit a Kaggle session
LR_BERT = 1e-5
LR_HEADS = 1e-4  # see PAPER_STRICT_LR below
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.02
LABEL_SMOOTHING = 0.0
GRAD_CLIP = 1.0

# Architecture retained from the validated v8 implementation.
GAT_LAYERS = 2
GAT_HEADS = 4
GAT_WINDOW = 5
HIERARCHY_LEVELS = 4
DROPOUT = 0.10

# Paper reports gloss batch size 256. During joint training only the
# unique candidate glosses for the current context batch are needed;
# this value controls the chunk size used by the gloss encoder.
GLOSS_BATCH_SIZE = 256

# Paper-faithful component switches used for ablation runs.
USE_CONTEXT_ENCODER = True
USE_GLOSS_ENCODER = True
USE_TEXT_HIERARCHY = True
USE_KG = True
USE_GAT = True
EARLY_STOP = False  # False = use the paper-reported maximum iteration budget exactly


USE_FREQUENCY_PRIOR = False
PRIOR_ALPHA_GRID = np.linspace(0.0, 0.40, 9)
RUN_OFFICIAL_EVAL = True
SAVE_CHECKPOINT = True
CHECKPOINT_PATH = Path("/kaggle/working/wsd_cao2024_final_best.pt")
RESULTS_CSV = Path("/kaggle/working/wsd_cao2024_results.csv")
AMP = DEVICE.type == "cuda"


# -------------------- v12 additions --------------------
MAX_LEN_CTX = 192            # context word-pieces (was 96: long benchmark sentences silently lost the target word)
MAX_CTX_WORDS = 110          # context is cropped to a window of words AROUND the target before tokenising
INIT_PROJ_IDENTITY = True    # start context/sense projections as identity so both towers share a space
USE_GRAD_CKPT = True         # gradient checkpointing (safe on a T4). False is faster but needs more memory
PAPER_STRICT_LR = False      # True = single LR 1e-5 for everything (exactly as reported in the paper)
LR_KG = 1e-3                 # learning rate of the 117k x 128 KG embedding H (rows are updated rarely)
SE07_TOLERANCE_POINTS = 2.0  # pick the latest epoch within this many points of the best SE07 accuracy (noise level)
TRAIN_TIME_BUDGET_SEC = int(10.25 * 3600)   # stop starting new epochs past this (Kaggle hard limit is 12h)
if PAPER_STRICT_LR:
    LR_HEADS = LR_KG = 1e-5
print("v12: LR_HEADS =", LR_HEADS, "| LR_KG =", LR_KG, "| identity init =", INIT_PROJ_IDENTITY, "| grad ckpt =", USE_GRAD_CKPT)

print("MAX_TRAIN_EXAMPLES =", MAX_TRAIN_EXAMPLES)
print("EXPECTED_SEMCOR_EXAMPLES =", EXPECTED_SEMCOR_EXAMPLES)
print("EPOCHS =", EPOCHS)
print("CONTEXT_BATCH_SIZE =", CONTEXT_BATCH_SIZE)
print("LR_BERT =", LR_BERT)


In [ ]:

# ============================================================
# 4. WordNet + SemCor
# ============================================================
import nltk
for resource in ["wordnet", "omw-1.4", "semcor"]:
    nltk.download(resource, quiet=True)

from nltk.corpus import wordnet as wn
from nltk.corpus import semcor

print("WordNet synsets:", len(list(wn.all_synsets())))
print("SemCor sentences:", len(semcor.tagged_sents(tag="sem")))


In [ ]:
# ============================================================
# 5. SemCor extraction  (v12 FIX: use the TAGGED lemma, keep true multiword expressions)
# ============================================================
# v10 bug: the target lemma was taken from syn.lemmas()[0] (first lemma of the gold
# synset), so for ~25% of examples the candidate set came from a different word
# (e.g. "said" was trained against the senses of "state"). The tagged lemma is
# label.name(). True MWEs (lemma contains "_") are kept as one "_"-joined token,
# which is the format the Raganato benchmark files use.
def extract_semcor_examples(max_examples=None, seed=42, include_mwe=True):
    examples = []
    stats = Counter()
    sentences = semcor.tagged_sents(tag="sem")

    for sent in tqdm(sentences, desc="Parsing SemCor"):
        words, items = [], []          # items[i] = (lemma, synset) or None

        def visit(node):
            if isinstance(node, str):
                words.append(str(node)); items.append(None)
                return
            try:
                label = node.label()
            except Exception:
                label = None
            try:
                leaves = node.leaves()
            except Exception:
                leaves = []

            syn, lemma = None, None
            if label is not None:
                try:
                    syn, lemma = label.synset(), label.name()
                except Exception:
                    syn, lemma = None, None

            if syn is not None and len(leaves) == 1:
                words.append(str(leaves[0])); items.append((lemma, syn))
                return
            if include_mwe and syn is not None and len(leaves) > 1 and "_" in lemma:
                words.append("_".join(str(x) for x in leaves)); items.append((lemma, syn))
                stats["mwe"] += 1
                return
            for child in node:
                visit(child)

        visit(sent)
        sentence = " ".join(words)

        for idx, it in enumerate(items):
            if it is None:
                continue
            lemma_raw, syn = it
            lemma_name = lemma_raw.lower()
            pos = "a" if syn.pos() == "s" else syn.pos()      # same mapping as the benchmark parser
            candidate_names = [c.name() for c in wn.synsets(lemma_name, pos=pos)]
            if syn.name() not in candidate_names:
                stats["dropped_gold_not_in_candidates"] += 1
                continue
            examples.append({
                "sentence": sentence,
                "target": lemma_name,
                "target_idx": idx,
                "pos": pos,
                "gold": syn.name(),
                "candidates": candidate_names,
            })

    rng = random.Random(seed)
    rng.shuffle(examples)
    if max_examples is not None:
        examples = examples[:max_examples]
    print("extraction stats:", dict(stats))
    return examples

train_examples = extract_semcor_examples(MAX_TRAIN_EXAMPLES)

print("Training examples:", len(train_examples))
print("Expected paper SemCor count:", EXPECTED_SEMCOR_EXAMPLES)
print("Unique target lemmas:", len(set(x["target"] for x in train_examples)))
print("Average candidate senses:",
      round(np.mean([len(x["candidates"]) for x in train_examples]), 2))
if len(train_examples) < 200000 and MAX_TRAIN_EXAMPLES is None:
    print("WARNING: extracted SemCor examples are substantially below the paper's 226,036 count.")


In [ ]:
# ============================================================
# 20. Obtain the canonical Raganato evaluation framework
# ============================================================
# v8 fixes the v7 404 failure. We first try the historical official
# framework archive, then fall back to a pinned mirror that contains
# the same Raganato Evaluation_Datasets and Scorer.java files.

import subprocess, shutil

RAGANATO_DIR = Path("/kaggle/working/WSD_Evaluation_Framework")
RAGANATO_ZIP = Path("/kaggle/working/WSD_Evaluation_Framework.zip")
RAGANATO_DIR.mkdir(parents=True, exist_ok=True)

RAGANATO_ARCHIVE_URLS = [
    "http://lcl.uniroma1.it/wsdeval/data/WSD_Evaluation_Framework.zip",
    "https://lcl.uniroma1.it/wsdeval/data/WSD_Evaluation_Framework.zip",
]

# Pinned Hugging Face mirror of the Raganato framework. The source
# contains Evaluation_Datasets/{ALL,semeval2007,...} and Scorer.java.
HF_RAGANATO_BASE = (
    "https://huggingface.co/Harsha909/consec-model-weights/resolve/"
    "bb8a8179a8715debec6ef1c2041a74af0bd4c388/"
    "data/WSD_Evaluation_Framework"
)

DATASET_LAYOUT = {
    "SE07": "Evaluation_Datasets/semeval2007/semeval2007",
    "SE2":  "Evaluation_Datasets/senseval2/senseval2",
    "SE3":  "Evaluation_Datasets/senseval3/senseval3",
    "SE13": "Evaluation_Datasets/semeval2013/semeval2013",
    "SE15": "Evaluation_Datasets/semeval2015/semeval2015",
    "ALL":  "Evaluation_Datasets/ALL/ALL",
}


def framework_is_valid(root=RAGANATO_DIR):
    required = [
        root / "Evaluation_Datasets" / "Scorer.java",
        root / "Evaluation_Datasets" / "ALL" / "ALL.data.xml",
        root / "Evaluation_Datasets" / "ALL" / "ALL.gold.key.txt",
    ]
    return all(p.exists() and p.stat().st_size > 100 for p in required)


def download_file(url, destination, retries=3):
    destination.parent.mkdir(parents=True, exist_ok=True)
    last = None
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "Mozilla/5.0 Kaggle-WSD-Reproduction"}
            )
            with urllib.request.urlopen(req, timeout=90) as response:
                data = response.read()
            if len(data) < 100:
                raise RuntimeError(f"Downloaded file is unexpectedly small: {len(data)} bytes")
            destination.write_bytes(data)
            return
        except Exception as e:
            last = e
            print(f"  attempt {attempt}/{retries} failed: {e}")
            time.sleep(2 * attempt)
    raise last


def try_official_archive():
    for url in RAGANATO_ARCHIVE_URLS:
        try:
            print("Trying official archive:", url)
            if RAGANATO_ZIP.exists():
                RAGANATO_ZIP.unlink()
            download_file(url, RAGANATO_ZIP, retries=2)
            with zipfile.ZipFile(RAGANATO_ZIP) as z:
                z.extractall("/kaggle/working/")

            # Locate the extracted framework regardless of archive root name.
            candidates = [
                p for p in Path("/kaggle/working").iterdir()
                if p.is_dir() and (p / "Evaluation_Datasets").exists()
            ]
            for c in candidates:
                if c.resolve() != RAGANATO_DIR.resolve():
                    if RAGANATO_DIR.exists() and any(RAGANATO_DIR.iterdir()):
                        shutil.rmtree(RAGANATO_DIR)
                    c.rename(RAGANATO_DIR)
                if framework_is_valid():
                    print("Official archive ready:", RAGANATO_DIR)
                    return True
        except Exception as e:
            print("Official archive unavailable:", e)
    return False


def obtain_from_pinned_mirror():
    print("Using pinned Raganato mirror fallback.")
    required_paths = [
        "Evaluation_Datasets/Scorer.java",
        "Evaluation_Datasets/ALL/ALL.data.xml",
        "Evaluation_Datasets/ALL/ALL.gold.key.txt",
        "Evaluation_Datasets/semeval2007/semeval2007.data.xml",
        "Evaluation_Datasets/semeval2007/semeval2007.gold.key.txt",
        "Evaluation_Datasets/senseval2/senseval2.data.xml",
        "Evaluation_Datasets/senseval2/senseval2.gold.key.txt",
        "Evaluation_Datasets/senseval3/senseval3.data.xml",
        "Evaluation_Datasets/senseval3/senseval3.gold.key.txt",
        "Evaluation_Datasets/semeval2013/semeval2013.data.xml",
        "Evaluation_Datasets/semeval2013/semeval2013.gold.key.txt",
        "Evaluation_Datasets/semeval2015/semeval2015.data.xml",
        "Evaluation_Datasets/semeval2015/semeval2015.gold.key.txt",
    ]

    for rel in required_paths:
        dest = RAGANATO_DIR / rel.removeprefix("Evaluation_Datasets/") if rel.startswith("Evaluation_Datasets/") else RAGANATO_DIR / rel
        # RAGANATO_DIR itself is the framework root, so keep Evaluation_Datasets.
        dest = RAGANATO_DIR / rel
        if dest.exists() and dest.stat().st_size > 100:
            continue
        url = HF_RAGANATO_BASE + "/" + rel
        print("Downloading:", rel)
        download_file(url, dest, retries=3)

    if not framework_is_valid():
        raise RuntimeError("Pinned mirror download completed but framework validation failed.")


if RUN_OFFICIAL_EVAL:
    if not framework_is_valid():
        got_archive = try_official_archive()
        if not got_archive:
            obtain_from_pinned_mirror()

    print("Raganato framework ready:", RAGANATO_DIR)
    print("Scorer.java:", RAGANATO_DIR / "Evaluation_Datasets" / "Scorer.java")


In [ ]:
# ============================================================
# 21. Parse benchmark datasets robustly
# ============================================================
from xml.etree import ElementTree as ET

# Raganato XML files may encode POS as either lowercase or uppercase.
# NLTK WordNet expects lowercase POS tags.
POS_MAP = {
    # WordNet single-letter POS tags
    "n": "n", "N": "n",
    "v": "v", "V": "v",
    "a": "a", "A": "a",
    "r": "r", "R": "r",
    "s": "a", "S": "a",
    # Raganato/SemEval XML sometimes uses full POS names
    "noun": "n", "NOUN": "n",
    "verb": "v", "VERB": "v",
    "adj": "a", "ADJ": "a",
    "adjective": "a", "ADJECTIVE": "a",
    "adv": "r", "ADV": "r",
    "adverb": "r", "ADVERB": "r",
    "sat": "a", "SAT": "a",
    "satellite": "a", "SATELLITE": "a",
}


def normalize_wordnet_pos(raw_pos):
    """Normalize every POS spelling encountered in Raganato XML to NLTK WordNet."""
    raw = (raw_pos or "").strip()
    if raw in POS_MAP:
        return POS_MAP[raw]
    low = raw.lower()
    if low in POS_MAP:
        return POS_MAP[low]
    # Defensive handling for tags such as VERB/NOUN with whitespace or
    # implementation-specific prefixes.
    if low.startswith("noun"):
        return "n"
    if low.startswith("verb"):
        return "v"
    if low.startswith("adj"):
        return "a"
    if low.startswith("adv"):
        return "r"
    if low.startswith("sat"):
        return "a"
    return None

DATASET_NAMES = ["SE07", "SE2", "SE3", "SE13", "SE15", "ALL"]


def dataset_prefix(name):
    return RAGANATO_DIR / DATASET_LAYOUT[name]


def find_dataset_files(name):
    prefix = dataset_prefix(name)
    xml = Path(str(prefix) + ".data.xml")
    gold = Path(str(prefix) + ".gold.key.txt")
    if not xml.exists() or not gold.exists():
        raise FileNotFoundError(
            f"Missing Raganato files for {name}: XML={xml.exists()}, GOLD={gold.exists()}"
        )
    return xml, gold


def load_gold_file(path):
    gold = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                gold[parts[0]] = parts[1:]
    return gold


def parse_raganato_dataset(name):
    xml_path, gold_path = find_dataset_files(name)
    gold_keys = load_gold_file(gold_path)
    root = ET.parse(xml_path).getroot()
    examples = []

    for text in root.findall(".//text"):
        for sentence in text.findall("./sentence"):
            words = []
            instances = []

            for child in list(sentence):
                token = (child.text or "").strip()
                if not token:
                    continue

                idx = len(words)
                words.append(token)

                if child.tag == "instance":
                    raw_pos = child.attrib.get("pos", "")
                    pos = normalize_wordnet_pos(raw_pos)
                    if pos is None:
                        raise ValueError(
                            f"Unsupported WordNet POS {raw_pos!r} in {xml_path} "
                            f"for instance {child.attrib.get('id')}. "
                            "Expected n/v/a/r or a standard full POS name."
                        )
                    instances.append({
                        "id": child.attrib["id"],
                        "idx": idx,
                        "lemma": child.attrib["lemma"].lower(),
                        "pos": pos,
                    })

            sentence_text = " ".join(words)

            for inst in instances:
                keys = gold_keys.get(inst["id"], [])
                if not keys:
                    continue

                gold_synsets = []
                for key in keys:
                    try:
                        gold_synsets.append(wn.lemma_from_key(key).synset().name())
                    except Exception:
                        pass

                candidates = [
                    s.name() for s in wn.synsets(inst["lemma"], pos=inst["pos"])
                ]

                gold_synsets = [s for s in gold_synsets if s in candidates]
                if not gold_synsets or not candidates:
                    continue

                examples.append({
                    "id": inst["id"],
                    "sentence": sentence_text,
                    "target": inst["lemma"],
                    "target_idx": inst["idx"],
                    "pos": inst["pos"],
                    "gold": gold_synsets[0],
                    "gold_all": gold_synsets,
                    "gold_keys": keys,
                    "candidates": candidates,
                })

    print(f"{name}: parsed {len(examples):,} instances")
    if len(examples) == 0:
        raise RuntimeError(
            f"{name} parsed zero instances. Check XML/gold compatibility and POS normalization."
        )
    return examples


if RUN_OFFICIAL_EVAL:
    benchmark = {
        name: parse_raganato_dataset(name)
        for name in DATASET_NAMES
    }

    print("\nBenchmark instance counts:")
    for name in DATASET_NAMES:
        print(f"  {name:5s}: {len(benchmark[name]):,}")

    component_sum = sum(len(benchmark[n]) for n in DATASET_NAMES[:-1])
    all_count = len(benchmark["ALL"])
    print("Sum of five component sets:", component_sum)
    print("Official ALL count:", all_count)
    if component_sum != all_count:
        print("NOTE: ALL is treated as the official framework's own file, not a manual concatenation.")


In [ ]:
# ============================================================
# 6. TEXT HIERARCHY + LOCAL 2-LAYER GAT
# ============================================================
def partition_correlation(word_vectors, lo, mid, hi):
    """Correlation/relevance between two newly created partitions.

    The paper's Algorithm 1 evaluates the relationship between the two
    partitions produced by each split. We operationalize that relation as
    cosine similarity between their mean contextual representations.
    """
    left = word_vectors[lo:mid].mean(dim=0)
    right = word_vectors[mid:hi].mean(dim=0)
    return F.cosine_similarity(left.unsqueeze(0), right.unsqueeze(0)).squeeze(0)


def build_text_hierarchy(word_vectors, max_steps=None):
    """Recursive partitioning (same algorithm and tie-breaking as before), but fast:
    the structure is chosen on a detached CPU copy with prefix sums, so there is ONE
    GPU->CPU sync per call instead of hundreds. The relevance links R now hold float correlations."""
    n = word_vectors.size(0)
    if n <= 1:
        return [], [[(0, n)]]
    if max_steps is None:
        max_steps = min(n - 1, 8)

    wv = word_vectors.detach().float().cpu()
    P = torch.cat([wv.new_zeros(1, wv.size(1)), wv.cumsum(0)])      # prefix sums, P[k] = sum of first k vectors

    active = [(0, n)]
    R = []
    H = [[(0, n)]]
    for _ in range(max_steps):
        best = None
        for lo, hi in active:
            if hi - lo <= 1:
                continue
            js = torch.arange(lo + 1, hi)
            left = (P[js] - P[lo]) / (js - lo).unsqueeze(1).float()
            right = (P[hi] - P[js]) / (hi - js).unsqueeze(1).float()
            cos = F.cosine_similarity(left, right, dim=1)
            v, k = cos.min(0)
            v = float(v)
            if best is None or v < best[0]:
                best = (v, lo, int(js[int(k)]), hi)
        if best is None:
            break
        corr, lo, j, hi = best
        active.remove((lo, hi))
        left_span, right_span = (lo, j), (j, hi)
        active.extend([left_span, right_span])
        R.append({"parent": (lo, hi), "split": j, "left": left_span,
                  "right": right_span, "correlation": corr})
        H.append(sorted(active))
    return R, H


def hierarchy_target_vector(word_vectors, target_idx):
    """Use all hierarchy partitions containing the target, with local
    partitions receiving greater weight as in the paper's hierarchy idea."""
    R, H = build_text_hierarchy(word_vectors)
    selected = []
    weights = []

    for level in H:
        for lo, hi in level:
            if lo <= target_idx < hi:
                width = hi - lo
                selected.append(word_vectors[lo:hi].mean(dim=0))
                weights.append(1.0 / math.sqrt(max(1, width)))

    if not selected:
        return word_vectors[target_idx]

    w = torch.tensor(weights, device=word_vectors.device,
                      dtype=word_vectors.dtype)
    w = w / w.sum()
    return torch.stack(selected).mul(w[:, None]).sum(dim=0)


class LocalGAT(nn.Module):
    """Paper Section 3.5: two-stage attention over contextual neighbors.

    This is deliberately a sentence-level local graph, not a full WordNet
    graph. Each layer computes attention coefficients between the target
    representation and its neighboring contextual word representations,
    normalizes them, and aggregates them. Multi-head attention is used.
    """
    def __init__(self, hidden, heads=4, layers=2, window=5, dropout=0.1):
        super().__init__()
        assert hidden % heads == 0
        self.hidden = hidden
        self.heads = heads
        self.head_dim = hidden // heads
        self.window = window
        self.proj = nn.ModuleList([
            nn.Linear(hidden, hidden, bias=False) for _ in range(layers)
        ])
        self.attn = nn.Parameter(torch.empty(layers, heads, 2 * self.head_dim))
        nn.init.xavier_uniform_(self.attn)
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.dropout = nn.Dropout(dropout)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, x, target_idx):
        y = x
        target = y[target_idx]

        for layer, (proj, norm) in enumerate(zip(self.proj, self.norms)):
            h = proj(y).view(y.size(0), self.heads, self.head_dim)
            lo = max(0, target_idx - self.window)
            hi = min(y.size(0), target_idx + self.window + 1)
            idx = torch.arange(lo, hi, device=y.device)

            q = h[target_idx]
            neigh = h[idx]
            q2 = q.unsqueeze(0).expand(neigh.size(0), -1, -1)
            e = self.leaky(
                (torch.cat([q2, neigh], dim=-1) *
                 self.attn[layer].unsqueeze(0)).sum(-1)
            )
            alpha = torch.softmax(e, dim=0)
            alpha = self.dropout(alpha)
            agg = (alpha.unsqueeze(-1) * neigh).sum(dim=0).reshape(self.hidden)
            target = norm(target + F.gelu(agg))
            y = y.clone()
            y[target_idx] = target

        return target


def crop_words(words, ti, max_words=None):
    """Keep a window of words around the target so truncation can never drop the target word."""
    max_words = max_words or MAX_CTX_WORDS
    if len(words) <= max_words:
        return words, ti
    lo = max(0, min(ti - max_words // 2, len(words) - max_words))
    return words[lo:lo + max_words], ti - lo


In [ ]:
# ============================================================
# 7. WORDNET 3.0 KNOWLEDGE GRAPH — ADJACENCY + LEARNABLE H
# ============================================================
# The paper constructs A from WordNet hypernym/hyponym relations and
# combines A with a learnable representation matrix H, a weight O and bias.
# A dense 117k x 117k matrix is unnecessary: we store the exact same
# adjacency as neighbor lists and use sparse/neighbor aggregation.

ALL_SYNSETS = list(wn.all_synsets())
SYNSET_NAMES = [s.name() for s in ALL_SYNSETS]
SENSE_TO_ID = {name: i for i, name in enumerate(SYNSET_NAMES)}

KG_NEIGHBORS = [[] for _ in ALL_SYNSETS]
KG_RELATIONS = [[] for _ in ALL_SYNSETS]

for i, syn in enumerate(ALL_SYNSETS):
    # Both directions are represented because the paper explicitly
    # considers both hypernym and hyponym relations.
    for n in syn.hypernyms():
        KG_NEIGHBORS[i].append(SENSE_TO_ID[n.name()])
        KG_RELATIONS[i].append("hypernym")
    for n in syn.hyponyms():
        KG_NEIGHBORS[i].append(SENSE_TO_ID[n.name()])
        KG_RELATIONS[i].append("hyponym")

# Remove duplicate neighbor IDs while retaining relation information.
for i in range(len(KG_NEIGHBORS)):
    pairs = list(dict.fromkeys(zip(KG_NEIGHBORS[i], KG_RELATIONS[i])))
    KG_NEIGHBORS[i] = [p[0] for p in pairs]
    KG_RELATIONS[i] = [p[1] for p in pairs]

print("WordNet synsets:", len(ALL_SYNSETS))
print("Directed hypernym/hyponym edges:", sum(len(x) for x in KG_NEIGHBORS))

# Explicit sparse adjacency matrix A. The neural forward pass below uses the
# equivalent row-wise neighbor operation instead of materializing A@H every
# batch, which is mathematically the same operation for each selected row.
from scipy.sparse import csr_matrix
_rows, _cols, _vals = [], [], []
for i, neigh in enumerate(KG_NEIGHBORS):
    for j in neigh:
        _rows.append(i); _cols.append(j); _vals.append(1.0)
A = csr_matrix(
    (np.asarray(_vals, dtype=np.float32),
     (np.asarray(_rows, dtype=np.int32), np.asarray(_cols, dtype=np.int32))),
    shape=(len(ALL_SYNSETS), len(ALL_SYNSETS))
)
print("Sparse adjacency A shape:", A.shape, "nnz:", A.nnz)

# The gloss text supplies the annotated semantic information H-side of the
# bi-encoder; the graph branch below supplies the structured Q vector.
def relation_aware_gloss_text(name):
    # The gloss encoder receives the WordNet sense gloss/definition.
    # Structured hypernym/hyponym information is supplied by the separate
    # KG branch rather than being appended to the text input.
    syn = wn.synset(name)
    return syn.definition()

print("KG example:", SYNSET_NAMES[0], "neighbors:", KG_NEIGHBORS[0][:5])


# ---- v12: CSR neighbour lists so the KG branch can be computed in one vectorised call ----
def _build_csr(rel_name):
    ptr, idx = [0], []
    for i in range(len(KG_NEIGHBORS)):
        for j, r in zip(KG_NEIGHBORS[i], KG_RELATIONS[i]):
            if r == rel_name:
                idx.append(j)
        ptr.append(len(idx))
    return np.asarray(ptr, dtype=np.int64), np.asarray(idx, dtype=np.int64)

HYPER_PTR, HYPER_IDX = _build_csr("hypernym")
HYPO_PTR, HYPO_IDX = _build_csr("hyponym")

def gather_csr(ptr, idx, ids):
    chunks = [idx[ptr[i]:ptr[i + 1]] for i in ids]
    counts = np.asarray([len(c) for c in chunks], dtype=np.int64)
    flat = np.concatenate(chunks) if len(chunks) and counts.sum() > 0 else np.zeros(0, dtype=np.int64)
    offsets = np.concatenate([[0], np.cumsum(counts)[:-1]]).astype(np.int64)
    return (torch.from_numpy(flat).to(DEVICE), torch.from_numpy(offsets).to(DEVICE),
            torch.from_numpy(counts).to(DEVICE).float())

print("KG CSR built: hypernym edges", len(HYPER_IDX), "| hyponym edges", len(HYPO_IDX))


In [ ]:
# ============================================================
# 8. BERT tokenizer + jointly trainable encoders
# ============================================================
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

context_encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
gloss_encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

# v9: BOTH encoders are trainable, matching the paper's bi-encoder setup.
for p in context_encoder.parameters():
    p.requires_grad = True
for p in gloss_encoder.parameters():
    p.requires_grad = True

# Gradient checkpointing reduces activation memory enough to run both BERTs
# jointly on a 14.56 GiB Tesla T4. It does not freeze either encoder.
try:
    if not USE_GRAD_CKPT:
        raise RuntimeError('disabled by USE_GRAD_CKPT=False')
    context_encoder.gradient_checkpointing_enable()
    gloss_encoder.gradient_checkpointing_enable()
    context_encoder.config.use_cache = False
    gloss_encoder.config.use_cache = False
    print("Gradient checkpointing: enabled for context + gloss BERT")
except Exception as e:
    print("Gradient checkpointing OFF:", e)

print("Context parameters:", sum(p.numel() for p in context_encoder.parameters()))
print("Gloss parameters trainable:", sum(p.numel() for p in gloss_encoder.parameters()))


In [ ]:
# ============================================================
# 9. Gloss encoding: trainable during training, cached only after training
# ============================================================
def _encode_gloss_texts(texts, encoder, batch_size=GLOSS_BATCH_SIZE, grad=False):
    outputs = []
    context_mgr = torch.enable_grad() if grad else torch.no_grad()

    with context_mgr:
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            enc = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=GLOSS_MAX_LEN
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}

            with torch.autocast(device_type="cuda", dtype=torch.float16,
                                enabled=AMP):
                out = encoder(**enc).last_hidden_state[:, 0, :]

            outputs.append(F.normalize(out.float(), dim=-1))

    return torch.cat(outputs, dim=0) if outputs else torch.empty((0, 768), device=DEVICE)


@torch.no_grad()
def encode_gloss_cache(synset_names, batch_size=GLOSS_BATCH_SIZE):
    """Cache gloss vectors AFTER training using the fine-tuned gloss BERT."""
    gloss_encoder.eval()
    cache = {}
    names = list(synset_names)
    texts = [relation_aware_gloss_text(n) for n in names]
    out = _encode_gloss_texts(texts, gloss_encoder, batch_size=batch_size, grad=False)

    for n, vec in zip(names, out):
        cache[n] = vec.detach().cpu()

    return cache

print("v9 gloss mode: trainable during main training; cache is created only after training.")


In [ ]:
# ============================================================
# 10. PAPER MODEL: KG + HIERARCHY + BERT BI-ENCODER + GAT
# ============================================================
class KGStructuredEncoder(nn.Module):
    """Implementation of the paper's adjacency-based KG branch.

    H: learnable vector for every WordNet synset.
    A*H: neighbor aggregation induced by the WordNet adjacency A.
    O,b_q: trainable projection to the BERT feature dimension.
    q_s: weighted self + relation-neighbor combination.
    """
    def __init__(self, n_nodes, hidden, kg_dim=128):
        super().__init__()
        self.node_h = nn.Embedding(n_nodes, kg_dim)
        self.O = nn.Linear(kg_dim, hidden)
        self.b_q = nn.Parameter(torch.zeros(hidden))
        self.hyper_weight = nn.Parameter(torch.tensor(1.0))
        self.hypo_weight = nn.Parameter(torch.tensor(1.0))
        nn.init.normal_(self.node_h.weight, mean=0.0, std=0.02)

    def forward(self, names):
        ids = [SENSE_TO_ID[n] for n in names]
        base = self.node_h(torch.tensor(ids, dtype=torch.long, device=DEVICE))

        # v12 FIX: v11/FINAL computed the relation weights with float(self.hyper_weight), which
        # detaches them (they could never learn) and forced a GPU sync per neighbour.
        # Same maths (weights normalised over all neighbours), now vectorised and differentiable.
        h_ids, h_off, n_h = gather_csr(HYPER_PTR, HYPER_IDX, ids)
        o_ids, o_off, n_o = gather_csr(HYPO_PTR, HYPO_IDX, ids)
        W = self.node_h.weight
        sum_h = F.embedding_bag(h_ids, W, h_off, mode="sum")
        sum_o = F.embedding_bag(o_ids, W, o_off, mode="sum")
        wh, wo = self.hyper_weight, self.hypo_weight
        denom = wh * n_h + wo * n_o + 1e-8
        neigh = (wh * sum_h + wo * sum_o) / denom.unsqueeze(1)

        q = base + neigh
        q = self.O(q) + self.b_q
        # Equation (3) / final KG normalization described in Section 3.2.
        return F.normalize(q, dim=-1)


class HighQualityWSDModel(nn.Module):
    def __init__(self, context_encoder, gloss_encoder, hidden=768):
        super().__init__()
        self.context_encoder = context_encoder
        self.gloss_encoder = gloss_encoder
        self.hidden = hidden
        self.kg_encoder = KGStructuredEncoder(len(ALL_SYNSETS), hidden, kg_dim=128)

        self.context_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
        )
        self.sense_proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
        )
        # Paper Section 3.4: gloss representation and normalized KG
        # representation are concatenated to form r_gs.
        self.sense_fusion = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.LayerNorm(hidden),
        )
        self.gat = LocalGAT(hidden, heads=GAT_HEADS, layers=GAT_LAYERS,
                            window=GAT_WINDOW, dropout=DROPOUT)
        self.hierarchy_gate = nn.Parameter(torch.tensor(0.5))
        self.gat_gate = nn.Parameter(torch.tensor(0.5))
        self.kg_gate = nn.Parameter(torch.tensor(0.5))
        self.log_temperature = nn.Parameter(torch.tensor(math.log(10.0)))

        # v12: the two towers must start in the SAME space. With default random init, context_proj and
        # sense_proj/sense_fusion are unrelated random maps, so the initial score is meaningless and the
        # heads have to learn the alignment from scratch at a tiny learning rate.
        if INIT_PROJ_IDENTITY:
            with torch.no_grad():
                for lin in (self.context_proj[0], self.sense_proj[0]):
                    lin.weight.copy_(torch.eye(hidden)); lin.bias.zero_()
                self.sense_fusion[0].weight.zero_()
                self.sense_fusion[0].weight[:, :hidden].copy_(torch.eye(hidden))   # [I | 0]: KG half starts silent
                self.sense_fusion[0].bias.zero_()

    def _word_vectors(self, hidden_states, word_ids, n_words):
        buckets = defaultdict(list)
        for j, wid in enumerate(word_ids):
            if wid is not None:
                buckets[wid].append(j)
        if not buckets:
            return hidden_states[0].unsqueeze(0)
        cls = hidden_states[0]
        # v12: a word that yields no word-piece keeps a placeholder, so later word indices never shift.
        return torch.stack([hidden_states[buckets[w]].mean(dim=0) if w in buckets else cls
                            for w in range(max(buckets) + 1)])

    def encode_context_batch(self, examples):
        word_batches, tis = [], []
        for ex in examples:
            w, ti = crop_words(ex["sentence"].split(), ex["target_idx"])
            word_batches.append(w); tis.append(ti)
        enc = tokenizer(
            word_batches, is_split_into_words=True, return_tensors="pt",
            truncation=True, max_length=MAX_LEN_CTX, padding=True
        )
        word_id_lists = [enc.word_ids(batch_index=i) for i in range(len(examples))]
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        if USE_CONTEXT_ENCODER:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                hidden_states = self.context_encoder(**enc).last_hidden_state
        else:
            # Ablation: no context encoder; use a fixed zero-independent
            # embedding so only the remaining branches determine the score.
            hidden_states = torch.zeros(
                enc["input_ids"].size(0), enc["input_ids"].size(1),
                self.hidden, device=DEVICE
            )

        contexts = []
        for b, ex in enumerate(examples):
            wv = self._word_vectors(hidden_states[b], word_id_lists[b],
                                    len(word_batches[b]))
            target_idx = min(tis[b], wv.size(0) - 1)
            # Equation (4): a phrase target is represented by the mean of
            # contextual vectors for all words in its target span.
            if "target_span" in ex and ex["target_span"] is not None:
                p0, p1 = ex["target_span"]
                p0 = max(0, min(p0, wv.size(0) - 1))
                p1 = max(p0 + 1, min(p1, wv.size(0)))
                target_repr = wv[p0:p1].mean(dim=0)
            else:
                target_repr = wv[target_idx]

            if USE_TEXT_HIERARCHY:
                hierarchy = hierarchy_target_vector(wv, target_idx)
            else:
                hierarchy = torch.zeros_like(wv[target_idx])

            if USE_GAT:
                gat = self.gat(wv, target_idx)
            else:
                gat = torch.zeros_like(wv[target_idx])

            a = torch.sigmoid(self.hierarchy_gate)
            g = torch.sigmoid(self.gat_gate)
            context = target_repr + a * hierarchy + g * gat
            contexts.append(self.context_proj(context))

        return torch.stack(contexts)

    def encode_unique_glosses_trainable(self, candidates):
        names = list(dict.fromkeys(candidates))
        texts = [relation_aware_gloss_text(n) for n in names]
        if USE_GLOSS_ENCODER:
            out = _encode_gloss_texts(
                texts, self.gloss_encoder,
                batch_size=GLOSS_BATCH_SIZE, grad=True
            )
        else:
            out = torch.zeros((len(names), self.hidden), device=DEVICE)
        return {n: v for n, v in zip(names, out)}

    def sense_vectors_from_batch_glosses(self, candidates, gloss_vectors):
        if USE_GLOSS_ENCODER:
            gloss = torch.stack([gloss_vectors[c] for c in candidates])
            gloss = self.sense_proj(gloss)
        else:
            gloss = torch.zeros((len(candidates), self.hidden), device=DEVICE)

        if USE_KG:
            kg = self.kg_encoder(candidates)
            k = torch.sigmoid(self.kg_gate)
            # Paper: r_gs is formed by splicing/concatenating the gloss
            # representation with the normalized KG representation.
            sense = self.sense_fusion(torch.cat([gloss, k * kg], dim=-1))
        else:
            sense = gloss

        return F.normalize(sense, dim=-1)

    def sense_vectors_cached(self, candidates, gloss_cache_cpu):
        if USE_GLOSS_ENCODER:
            gloss = torch.stack([gloss_cache_cpu[c] for c in candidates]).to(DEVICE)
            gloss = self.sense_proj(gloss)
        else:
            gloss = torch.zeros((len(candidates), self.hidden), device=DEVICE)

        if USE_KG:
            kg = self.kg_encoder(candidates)
            k = torch.sigmoid(self.kg_gate)
            sense = self.sense_fusion(torch.cat([gloss, k * kg], dim=-1))
        else:
            sense = gloss

        return F.normalize(sense, dim=-1)

    def score_batch(self, examples, gloss_cache_cpu=None, training_joint=False):
        contexts = F.normalize(self.encode_context_batch(examples), dim=-1)

        gloss_vectors = None
        if training_joint:
            all_candidates = [c for ex in examples for c in ex["candidates"]]
            gloss_vectors = self.encode_unique_glosses_trainable(all_candidates)

        losses, all_scores = [], []
        temp = self.log_temperature.exp().clamp(1.0, 30.0)

        for i, ex in enumerate(examples):
            if training_joint:
                senses = self.sense_vectors_from_batch_glosses(
                    ex["candidates"], gloss_vectors
                )
            else:
                senses = self.sense_vectors_cached(ex["candidates"], gloss_cache_cpu)

            scores = temp * torch.matmul(senses, contexts[i])
            target = torch.tensor(
                ex["candidates"].index(ex["gold"]), device=DEVICE
            )
            losses.append(F.cross_entropy(scores.unsqueeze(0), target.unsqueeze(0)))
            all_scores.append(scores)

        return torch.stack(losses).mean(), all_scores

    def score_example(self, ex, gloss_cache_cpu):
        _, scores = self.score_batch([ex], gloss_cache_cpu, training_joint=False)
        return scores[0]


model = HighQualityWSDModel(context_encoder, gloss_encoder).to(DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# ============================================================
# 12. Optimizer, scheduler, and joint batched training
# ============================================================
def optimizer_and_scheduler(model, total_steps):
    bert_params, kg_params, head_params = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("context_encoder.") or name.startswith("gloss_encoder."):
            bert_params.append(p)
        elif name == "kg_encoder.node_h.weight":
            kg_params.append(p)          # 117k x 128 embedding: rows are touched rarely, needs a larger LR
        else:
            head_params.append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": bert_params, "lr": LR_BERT},
            {"params": kg_params, "lr": LR_KG, "weight_decay": 0.0},
            {"params": head_params, "lr": LR_HEADS},
        ],
        weight_decay=WEIGHT_DECAY
    )

    warmup = max(1, int(total_steps * WARMUP_RATIO))
    def lr_lambda(step):
        if step < warmup:
            return float(step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def train_epoch(model, examples, optimizer, scheduler, scaler):
    model.train()
    order = list(range(len(examples)))
    random.shuffle(order)
    total, n_ok, n_oom = 0.0, 0, 0

    for start in tqdm(range(0, len(order), CONTEXT_BATCH_SIZE), desc="Joint training"):
        batch = [examples[i] for i in order[start:start + CONTEXT_BATCH_SIZE]]
        optimizer.zero_grad(set_to_none=True)
        try:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                loss, _ = model.score_batch(batch, training_joint=True)
            scaler.scale(loss).backward()          # v12: GradScaler (fp16 autocast without it can underflow gradients)
        except torch.cuda.OutOfMemoryError:
            n_oom += 1
            optimizer.zero_grad(set_to_none=True)
            loss = None
            torch.cuda.empty_cache()
            scheduler.step()
            continue
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total += float(loss.detach().cpu())
        n_ok += 1
        if n_ok % 500 == 0:
            print(f"batch={n_ok:6d} loss={total/n_ok:.4f} bert_lr={optimizer.param_groups[0]['lr']:.2e}"
                  + (f" oom_skipped={n_oom}" if n_oom else ""))
    if n_oom:
        print(f"WARNING: {n_oom} batches skipped (CUDA OOM). Lower MAX_CTX_WORDS / GLOSS_MAX_LEN if this is large.")
    return total / max(1, n_ok)


In [ ]:
# ============================================================
# 13. Development evaluation
# ============================================================
@torch.no_grad()
def predict_raw(model, ex, gloss_cache_cpu):
    model.eval()
    if len(ex["candidates"]) == 1:
        return ex["candidates"][0]
    scores = model.score_example(ex, gloss_cache_cpu)
    return ex["candidates"][int(scores.argmax())]


@torch.no_grad()
def evaluate_local(model, examples, gloss_cache_cpu, limit=None):
    """v12: accuracy against ALL gold senses (the scorer accepts any gold key). The old macro-F1 over
    synset classes was not a meaningful dev metric and has been dropped."""
    model.eval()
    subset = examples if limit is None else examples[:limit]
    correct = 0
    for ex in tqdm(subset, desc="Development evaluation"):
        correct += predict_raw(model, ex, gloss_cache_cpu) in ex["gold_all"]
    return {"accuracy": correct / max(1, len(subset)), "instances": len(subset)}


In [ ]:
# ============================================================
# 14. MAIN PAPER TRAINING: SEMCOR TRAIN / SE07 DEVELOPMENT
# ============================================================
# v12 changes:
#  * FIX: the SE07 gloss cache is rebuilt with the CURRENT gloss encoder every epoch. Before, it was built once
#    from the untrained gloss BERT, so every per-epoch dev score (and the "best epoch" choice) was wrong.
#  * every epoch is checkpointed to disk (a killed session no longer loses everything);
#  * final model = latest epoch within SE07_TOLERANCE_POINTS of the best (SE07 has 455 items, so +-2 points is noise);
#  * wall-clock guard for the 12h Kaggle limit.
train_pool = train_examples
dev_examples = parse_raganato_dataset("SE07")
dev_senses = sorted(set(c for ex in dev_examples for c in ex["candidates"]))

print("SemCor train examples:", len(train_pool))
print("Official SE07 development instances:", len(dev_examples))
print("Paper-reported SemCor examples:", EXPECTED_SEMCOR_EXAMPLES)

steps_per_epoch = math.ceil(len(train_pool) / CONTEXT_BATCH_SIZE)
total_steps = EPOCHS * steps_per_epoch
optimizer, scheduler = optimizer_and_scheduler(model, total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=AMP)

EPOCH_CKPT_DIR = CHECKPOINT_PATH.parent / "epoch_ckpts"
EPOCH_CKPT_DIR.mkdir(parents=True, exist_ok=True)
history, ckpt_paths = [], []
t_start = time.time()

for epoch in range(EPOCHS):
    t_ep = time.time()
    print(f"\n{'='*75}\nITERATION {epoch+1}/{EPOCHS}\n{'='*75}")
    train_loss = train_epoch(model, train_pool, optimizer, scheduler, scaler)

    dev_gloss_cache = encode_gloss_cache(dev_senses, GLOSS_BATCH_SIZE)      # <- rebuilt every epoch
    dev_metrics = evaluate_local(model, dev_examples, dev_gloss_cache)
    row = {"iteration": epoch + 1, "train_loss": train_loss,
           "SE07_accuracy": 100.0 * dev_metrics["accuracy"]}
    history.append(row)
    print(row)

    path = EPOCH_CKPT_DIR / f"epoch{epoch+1}.pt"
    torch.save(model.state_dict(), path)
    ckpt_paths.append(path)

    elapsed, ep_time = time.time() - t_start, time.time() - t_ep
    print(f"Epoch time {ep_time/3600:.2f} h | elapsed {elapsed/3600:.2f} h | budget {TRAIN_TIME_BUDGET_SEC/3600:.2f} h")
    if epoch + 1 < EPOCHS and elapsed + ep_time * 1.05 > TRAIN_TIME_BUDGET_SEC:
        print("Stopping early: the next epoch would exceed the wall-clock budget "
              "(LR schedule not fully annealed; lower EPOCHS next time).")
        break

accs = [h["SE07_accuracy"] for h in history]
best_acc = max(accs)
chosen = max(i for i, a in enumerate(accs) if a >= best_acc - SE07_TOLERANCE_POINTS)
best_dev = accs[chosen] / 100.0
print(f"Best SE07 = {best_acc:.2f}; selected iteration {chosen+1} (latest within {SE07_TOLERANCE_POINTS} points of best)")
model.load_state_dict(torch.load(ckpt_paths[chosen], map_location=DEVICE))

if SAVE_CHECKPOINT:
    torch.save({
        "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "selected_iteration": chosen + 1,
        "config": {
            "paper": "Cao et al. 2024",
            "model_name": MODEL_NAME,
            "seed": SEED,
            "epochs_run": len(history),
            "max_iterations_in_paper": MAX_ITERATIONS,
            "context_batch_size": CONTEXT_BATCH_SIZE,
            "gloss_batch_size": GLOSS_BATCH_SIZE,
            "lr_bert": LR_BERT, "lr_heads": LR_HEADS, "lr_kg": LR_KG,
            "gat_layers": GAT_LAYERS,
            "gat_heads": GAT_HEADS,
            "hierarchy": True,
            "wordnet_relations": "hypernym+hyponym",
            "main_result_frequency_prior": False,
        }
    }, CHECKPOINT_PATH)
    print("Saved:", CHECKPOINT_PATH)

history_df = pd.DataFrame(history)
display(history_df)


In [ ]:
# ============================================================
# 16. Joint checkpoint status
# ============================================================
print("Gloss encoder: TRAINABLE and jointly fine-tuned.")
print("Context encoder: TRAINABLE and fine-tuned.")
print("Best development checkpoint:", CHECKPOINT_PATH)


In [ ]:
# ============================================================
# 18. No frequency-prior calibration in the main reproduction
# ============================================================
# Kept as an explicit notebook section because earlier experimental
# notebooks used this post-processing step. It is intentionally disabled
# for the paper-faithful main result.
print("Frequency-prior post-processing: DISABLED for the main reproduction.")

In [ ]:
# ============================================================
# 22. Benchmark gloss cache + batched inference
# ============================================================
def extend_gloss_cache(model, examples):
    names = sorted(set(c for ex in examples for c in ex["candidates"]))
    cache = encode_gloss_cache(names, batch_size=GLOSS_BATCH_SIZE)
    print("Benchmark gloss cache:", len(cache))
    return cache


BENCHMARK_BATCH_SIZE = 16

@torch.no_grad()
def predict_benchmark_batched(model, examples, gloss_cache_cpu,
                              batch_size=BENCHMARK_BATCH_SIZE, **_ignored):
    # **_ignored: the evaluation cell passes use_prior/alpha; the frequency prior is disabled in this notebook.
    model.eval()
    predictions = []

    for start in tqdm(
        range(0, len(examples), batch_size),
        desc="Official benchmark inference"
    ):
        batch = examples[start:start + batch_size]
        contexts = F.normalize(model.encode_context_batch(batch), dim=-1)
        temp = model.log_temperature.exp().clamp(1.0, 30.0)

        for i, ex in enumerate(batch):
            if len(ex["candidates"]) == 1:             # monosemous: nothing to score
                predictions.append(ex["candidates"][0])
                continue
            senses = model.sense_vectors_cached(ex["candidates"], gloss_cache_cpu)
            scores = temp * torch.matmul(senses, contexts[i])
            predictions.append(ex["candidates"][int(scores.argmax())])

    return predictions


def synset_to_sense_key(synset_name, target_lemma):
    """Robustly map a predicted WordNet synset to a Raganato sense key."""
    syn = wn.synset(synset_name)
    target_norm = str(target_lemma).strip().lower().replace(" ", "_")

    def norm(x):
        return str(x).lower().strip().replace(" ", "_")

    # 1. Exact lemma match.
    for lemma in syn.lemmas():
        if norm(lemma.name()) == target_norm:
            return lemma.key()

    # 2. WordNet morphology using the predicted synset POS.
    pos = syn.pos()
    morph = wn.morphy(target_norm, pos)
    if morph:
        for lemma in syn.lemmas():
            if norm(lemma.name()) == norm(morph):
                return lemma.key()

    # 3. Defensive morphology over all WordNet POS values.
    for pos_try in ("n", "v", "a", "r", "s"):
        morph = wn.morphy(target_norm, pos_try)
        if morph:
            for lemma in syn.lemmas():
                if norm(lemma.name()) == norm(morph):
                    return lemma.key()

    # 4. Candidate-synset identity fallback.
    # Benchmark candidates are generated from the surface target.
    # If the predicted synset is one of those candidates, its stored
    # WordNet lemma key is valid even for inflected targets such as
    # feelings -> feeling and cars -> car.
    candidate_names = set()
    for pos_try in (pos, "n", "v", "a", "r", "s"):
        try:
            candidate_names.update(
                x.name() for x in wn.synsets(target_norm, pos=pos_try)
            )
        except Exception:
            pass

    if synset_name in candidate_names and syn.lemmas():
        return syn.lemmas()[0].key()

    raise KeyError(
        f"Cannot map synset {synset_name} to target lemma "
        f"{target_lemma!r}; synset lemmas="
        f"{[l.name() for l in syn.lemmas()]}; "
        f"target candidate synsets={sorted(candidate_names)[:20]}"
    )

def write_prediction_file(examples, predictions, path):
    if len(examples) != len(predictions):
        raise ValueError("Prediction count does not match benchmark instance count.")

    seen = set()
    with open(path, "w", encoding="utf-8") as f:
        for ex, pred in zip(examples, predictions):
            if ex["id"] in seen:
                raise ValueError(f"Duplicate instance id: {ex['id']}")
            seen.add(ex["id"])
            key = synset_to_sense_key(pred, ex["target"])
            if not key or not str(key).strip():
                raise ValueError(
                    f"Empty sense key for instance {ex['id']}, "
                    f"target={ex['target']!r}, prediction={pred!r}"
                )
            f.write(f"{ex['id']} {key}\n")

    print("Saved predictions:", path, "lines:", len(predictions))


def python_fallback_score(examples, predictions):
    """Fallback only: exact-match accuracy against all gold keys."""
    correct = sum(
        pred in ex["gold_all"]
        for ex, pred in zip(examples, predictions)
    )
    return 100.0 * correct / max(1, len(examples))


# ============================================================
# 22b. PRE-FLIGHT: Raganato -> WordNet sense-key mapping
# ============================================================
_preflight_cases = [
    ("feeling.n.01", "feelings"),
    ("car.n.01", "cars"),
    ("run.v.01", "running"),
]
for _syn, _target in _preflight_cases:
    try:
        _key = synset_to_sense_key(_syn, _target)
        assert isinstance(_key, str) and _key.strip()
        print(f"PASS: {_target!r} -> {_syn} -> {_key}")
    except Exception as _e:
        raise RuntimeError(
            f"Sense-key preflight failed for target={_target!r}, "
            f"synset={_syn!r}: {_e}"
        )
print("SENSE-KEY PREFLIGHT PASSED.")


In [ ]:
# ============================================================
# 22b. PRE-FLIGHT: Raganato -> WordNet sense-key mapping
# ============================================================
# Runs before expensive benchmark inference.
_preflight_cases = [
    ("feeling.n.01", "feelings"),
    ("car.n.01", "cars"),
    ("run.v.01", "running"),
]
for _syn, _target in _preflight_cases:
    try:
        _key = synset_to_sense_key(_syn, _target)
        assert isinstance(_key, str) and _key.strip()
        print(f"PASS: {_target!r} -> {_syn} -> {_key}")
    except Exception as _e:
        raise RuntimeError(
            f"Sense-key preflight failed for target={_target!r}, "
            f"synset={_syn!r}: {_e}"
        )
print("SENSE-KEY PREFLIGHT PASSED.")


In [ ]:
# ============================================================
# 23. Official benchmark: generate keys + run Scorer.java
# ============================================================
PAPER_RESULTS = {
    "SE07": 74.3,
    "SE2": 79.1,
    "SE3": 77.0,
    "SE13": 79.6,
    "SE15": 79.9,
    "ALL": 78.5,
}

PREDICTION_DIR = Path("/kaggle/working/wsd_predictions")
SCORER_DIR = RAGANATO_DIR / "Evaluation_Datasets"
SCORER_JAVA = SCORER_DIR / "Scorer.java"
SCORER_LOG_DIR = Path("/kaggle/working/wsd_scorer_logs")
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
SCORER_LOG_DIR.mkdir(parents=True, exist_ok=True)


def compile_official_scorer():
    if not SCORER_JAVA.exists():
        return False, "Scorer.java not found"

    try:
        result = subprocess.run(
            ["javac", str(SCORER_JAVA)],
            cwd=str(SCORER_DIR),
            capture_output=True,
            text=True,
            timeout=120,
        )
        if result.returncode != 0:
            return False, result.stderr.strip()
        return True, "compiled"
    except FileNotFoundError:
        return False, "Java/JDK is not available in this Kaggle runtime"
    except Exception as e:
        return False, str(e)


def run_official_scorer(name, prediction_path):
    gold_path = find_dataset_files(name)[1]
    try:
        result = subprocess.run(
            ["java", "-cp", str(SCORER_DIR), "Scorer",
             str(gold_path), str(prediction_path)],
            cwd=str(SCORER_DIR),
            capture_output=True,
            text=True,
            timeout=180,
        )
        output = (result.stdout or "") + ("\n" + result.stderr if result.stderr else "")
        (SCORER_LOG_DIR / f"{name}_scorer.txt").write_text(output, encoding="utf-8")

        if result.returncode != 0:
            return {
                "ok": False,
                "F1": np.nan,
                "precision": np.nan,
                "recall": np.nan,
                "scorer_output": output,
                "error": f"Scorer exited with code {result.returncode}",
            }

        # Official scorer output is typically of the form:
        # DATASET P= xx.x% R= xx.x% F1= xx.x%
        scorer_names = {
            "SE07": "semeval2007",
            "SE2": "senseval2",
            "SE3": "senseval3",
            "SE13": "semeval2013",
            "SE15": "semeval2015",
            "ALL": "ALL",
        }
        scorer_label = scorer_names[name]
        pattern = re.compile(
            rf"(?:^|\n)\s*{re.escape(scorer_label)}\s+"
            rf"P\s*=\s*([0-9.]+)%\s+"
            rf"R\s*=\s*([0-9.]+)%\s+"
            rf"F1\s*=\s*([0-9.]+)%",
            flags=re.IGNORECASE,
        )
        match = pattern.search(output)

        # Some scorer versions print only the requested file's metrics under ALL.
        if match is None:
            generic = re.search(
                r"P\s*=\s*([0-9.]+)%\s+R\s*=\s*([0-9.]+)%\s+F1\s*=\s*([0-9.]+)%",
                output,
                flags=re.IGNORECASE,
            )
            match = generic

        if match is None:
            return {
                "ok": False,
                "F1": np.nan,
                "precision": np.nan,
                "recall": np.nan,
                "scorer_output": output,
                "error": "Could not parse F1 from Scorer.java output",
            }

        return {
            "ok": True,
            "precision": float(match.group(1)),
            "recall": float(match.group(2)),
            "F1": float(match.group(3)),
            "scorer_output": output,
            "error": "",
        }

    except FileNotFoundError:
        return {
            "ok": False,
            "F1": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "scorer_output": "",
            "error": "Java runtime is not available",
        }
    except Exception as e:
        return {
            "ok": False,
            "F1": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "scorer_output": "",
            "error": str(e),
        }



official_results = pd.DataFrame()

if RUN_OFFICIAL_EVAL:
    # Cache every candidate gloss needed by the six official datasets.
    all_bench_examples = [ex for name in DATASET_NAMES for ex in benchmark[name]]
    gloss_cache_cpu = extend_gloss_cache(model, all_bench_examples)

    rows = []
    predictions_by_dataset = {}

    # -------------------- RAW NEURAL --------------------
    for name in DATASET_NAMES:
        examples = benchmark[name]
        preds = predict_benchmark_batched(
            model, examples, gloss_cache_cpu,
            use_prior=False, alpha=0.0
        )
        predictions_by_dataset[("Neural", name)] = preds
        pred_path = PREDICTION_DIR / f"{name}_neural.key.txt"
        write_prediction_file(examples, preds, pred_path)

    scorer_ok, scorer_msg = compile_official_scorer()
    print("Official scorer:", scorer_msg)

    for name in DATASET_NAMES:
        examples = benchmark[name]
        preds = predictions_by_dataset[("Neural", name)]
        pred_path = PREDICTION_DIR / f"{name}_neural.key.txt"
        official = run_official_scorer(name, pred_path) if scorer_ok else {
            "ok": False, "F1": np.nan, "precision": np.nan,
            "recall": np.nan, "scorer_output": "", "error": scorer_msg
        }
        fallback_acc = python_fallback_score(examples, preds)
        rows.append({
            "Model": "Neural",
            "Dataset": name,
            "instances": len(examples),
            "Precision": official["precision"],
            "Recall": official["recall"],
            "F1": official["F1"],
            "Fallback_exact_match": fallback_acc,
            "Paper F1": PAPER_RESULTS[name],
            "Difference": official["F1"] - PAPER_RESULTS[name] if np.isfinite(official["F1"]) else np.nan,
            "Scorer status": "official" if official["ok"] else "fallback/error",
            "Error": official["error"],
        })

    official_results = pd.DataFrame(rows)
    display(official_results)
    official_results.to_csv(RESULTS_CSV, index=False)
    print("Saved results:", RESULTS_CSV)
    print("Prediction directory:", PREDICTION_DIR)
    print("Scorer logs:", SCORER_LOG_DIR)


In [ ]:
# ============================================================
# 25. Final compact report + artifact manifest
# ============================================================
if not official_results.empty:
    final_report = official_results[
        ["Model", "Dataset", "instances", "Precision", "Recall", "F1",
         "Fallback_exact_match", "Paper F1", "Difference", "Scorer status"]
    ].copy()

    for col in ["Precision", "Recall", "F1", "Fallback_exact_match", "Paper F1", "Difference"]:
        final_report[col] = final_report[col].round(2)

    display(final_report)

    print("\nV12 PAPER-FAITHFUL REPORT")
    print("Both BERT encoders were jointly fine-tuned on the full extracted SemCor pool. Gloss text = WordNet definition; WordNet structure enters through the learnable KG branch.")
    print("Saved artifacts:")
    print("Checkpoint:", CHECKPOINT_PATH)
    print("Results CSV:", RESULTS_CSV)
    print("Predictions:", PREDICTION_DIR)
    print("Scorer logs:", SCORER_LOG_DIR)
    print("Raganato framework:", RAGANATO_DIR)


In [ ]:
# ============================================================
# 27. OPTIONAL TRUE ABLATIONS — DISABLED BY DEFAULT
# ============================================================
# The paper's Table 5 ablations are retrained models, not merely inference
# switches. Running all six full models would multiply the Kaggle compute
# budget, so this cell is deliberately opt-in.
#
# Set RUN_TRUE_ABLATIONS = True only when you have completed the main model
# and want to reproduce Table 5 one ablation at a time.
RUN_TRUE_ABLATIONS = False

PAPER_ABLATION_RESULTS = pd.DataFrame([
    ["Full", 74.3, 79.1, 77.0, 79.6, 79.9, 78.5],
    ["W/O Context Encoder", 70.5, 75.3, 72.5, 74.6, 73.8, 75.3],
    ["W/O Gloss Encoder", 66.1, 70.4, 69.2, 69.5, 67.4, 68.3],
    ["W/O Context Hierarchical structure", 73.8, 78.9, 76.5, 79.1, 79.2, 77.0],
    ["W/O Knowledge Graph Embedding", 73.5, 78.2, 76.3, 78.5, 78.8, 77.9],
    ["W/O GAT", 73.4, 78.1, 76.4, 77.9, 78.4, 77.6],
], columns=["Model","SE07","SE2","SE3","SE13","SE15","ALL"])

print("Paper Table 5 reference:")
display(PAPER_ABLATION_RESULTS)

if RUN_TRUE_ABLATIONS:
    raise NotImplementedError(
        "True ablation retraining is intentionally gated. Run one ablation "
        "at a time by changing the component switch at the top of the model "
        "section and rerunning the training/evaluation cells. This prevents "
        "accidentally spending 5x the Kaggle budget."
    )